## Load parquet and join with audio

In [1]:
import pandas as pd

df = pd.read_parquet("/home/vladislavstankov/Projects/Python/overlap-annotations/predictions_turbo.parquet")
print(f"Parquet rows: {len(df)}")


Parquet rows: 15065


In [2]:
import pickle
import pandas as pd


with open("../pred_audio_mapping.pkl", "rb") as f:
    audio_mapping = pickle.load(f)

# ключ — (origin, start_overlap, end_overlap)
df["audio"] = [
    audio_mapping[(row.origin, row.start_overlap, row.end_overlap)]
    for _, row in df.iterrows()
]

In [3]:
df.head()

,start_overlap,end_overlap,Duration,origin,overlap_dur,rms_global,p80,prediction,is_bad,audio
0,372.210526,374.404075,2.193548,2013112709581012,0.036559,0.011394,0.013881,Tak jenom poslancu přinesli.,False,"[-0.0052109878, -0.0008518262, 0.0017182757, 0..."
1,687.069610,688.196944,1.127334,2013112721082122,0.018789,0.037645,0.047582,Vyčer.,False,"[0.011057173, 0.011022873, 0.007706336, 0.0032..."
2,584.067912,585.229202,1.161290,2013120410081022,0.047991,0.040322,0.055045,Priršuji tady.,False,"[-0.0018332701, -0.0023932147, 0.00027356108, ..."
3,518.845501,519.966044,1.120543,2013120610381052,0.018676,0.046632,0.062473,"Pro pana kolegova,",False,"[0.0033295206, -0.0026074317, -0.0010380978, 0..."
4,206.000000,207.460102,1.460102,2013120612081222,0.760611,0.004248,0.005131,Tak přeštěky rána.,True,"[-3.676518e-05, 0.00023124718, 0.002357181, 0...."


In [10]:
df[    (df.prediction.str.len() > 25)
    & (df.Duration > 3)
    & (~df.is_bad)
    & (~df.prediction.str.lower().str.contains("titulky"))].to_parquet("../annotation_pool.parquet")

## Load predictions and visualize samples

In [7]:
(
    df[(df.prediction.str.len() > 25) & (df.Duration > 3) & (~df.is_bad) & (~df.prediction.str.lower().str.contains("titulky"))].Duration.sum() / 3600,  
    df[df.Duration > 2].Duration.sum() / 3600,  
    df.Duration.sum() / 3600
)

(np.float64(2.853021271458214),
 np.float64(5.090908462554234),
 np.float64(9.076006557253345))

In [11]:
import IPython.display as ipd
from IPython.display import display, HTML

# Join predictions with audio from the already-loaded df
# df doesn't have audio, so we match by original parquet columns
# Use a simple positional merge: pick 10 random samples that exist in both
sample_indices = df[
    (df.prediction.str.len() > 25)
    & (df.Duration > 3)
    & (~df.is_bad)
    & (~df.prediction.str.lower().str.contains("titulky"))
].sample(5).index

for i, idx in enumerate(sample_indices):
    row = df.iloc[idx]
    prediction = row.get("prediction", "")
    prediction = prediction.replace("<sc>", "[sc]")

    # Find matching audio in the audio_map by the original df index if available
    # df was saved without original index, so we try to match by mp3 + start/end
    mp3 = row.get("mp3", "?")
    start = row.get("start_overlap", 0)
    end = row.get("end_overlap", 0)


    display(HTML(f"<h3>Sample {i+1}</h3>"))
    display(HTML(f"<b>File:</b> {row['origin']}<br>"
                 f"<b>Overlap:</b> {start:.2f}s – {end:.2f}s "
                 f"(dur: {end - start:.2f}s)<br>"
                 f"<b>Prediction:</b> {prediction}"))

    audio = row["audio"]
    display(ipd.Audio(audio, rate=16_000))


    display(HTML("<hr>"))

In [2]:
import pandas as pd
anotations_pool = pd.read_parquet("../annotation_pool.parquet")

In [3]:
anotations_pool

,start_overlap,end_overlap,Duration,origin,overlap_dur,rms_global,p80,prediction,is_bad,audio
18,717.127334,721.134126,4.006791,2014021211581212,0.094907,0.042391,0.051567,"Nakonec, včera jsme byli svědky při rozsáhlé d...",False,"[0.0036263233, 0.003235546, 0.0016517313, 0.00..."
38,613.521222,617.803056,4.281834,2014050710281042,0.169496,0.036923,0.052248,"Já jsem, kolegyně a kolegové.",False,"[0.019946145, 0.013020596, 0.0049663493, -0.00..."
43,342.074703,346.672326,4.597623,2014051417381752,0.094454,0.036892,0.046102,pak vyvěduje ještě přilážku ministra zahraničí...,False,"[-0.060114205, -0.056801047, -0.05165762, -0.0..."
54,361.704584,367.728353,6.023769,2014061309080922,0.100396,0.024288,0.020187,"Ano, takže by to byl druhý bod.",False,"[-0.0041889525, -0.004372181, -0.002629727, -0..."
59,604.000000,607.803056,3.803056,2014061909280942,0.063384,0.034029,0.047387,Ježíte potom v podrobné rozpravě přečtu na obr...,False,"[0.03757366, 0.043231048, 0.03745228, 0.027941..."
...,...,...,...,...,...,...,...,...,...,...
14987,326.000000,329.633277,3.633277,2023031010181032,0.214601,0.064477,0.071895,"Ano, ano. Nechme to tak. Děkuji.",False,"[-0.080206886, 0.027639749, -0.042389005, -0.0..."
15013,46.000000,57.521222,11.521222,2023051816481702,0.208998,0.100649,0.131199,"My se tady nebavíme o dětech, které jsou jako ...",False,"[-0.2086652, -0.23270859, -0.25878608, -0.2849..."
15024,721.629881,730.010187,8.380306,2023060213481402,0.174307,0.239944,0.295192,"A to v oblasti. Už je konec. Já si omlouvám, ž...",False,"[0.09031591, 0.12273258, 0.09885492, 0.0998066..."
15031,284.237691,287.534805,3.297114,2023061412081222,0.126429,0.136076,0.183570,"kritizující úřad, musím vás představit.",False,"[-0.07016802, -0.09376298, -0.122021176, -0.13..."


In [4]:
import soundfile as sf
import os

output_dir = "/home/vladislavstankov/Projects/Python/overlap-annotations/selected_audios"
os.makedirs(output_dir, exist_ok=True)

audio_paths = []
for idx, row in anotations_pool.iterrows():
    filename = f"{idx:06d}.wav"
    filepath = os.path.join(output_dir, filename)
    sf.write(filepath, row["audio"], samplerate=16000)
    audio_paths.append(filepath)

anotations_pool["audio_path"] = audio_paths
print(f"Saved {len(audio_paths)} audio files to {output_dir}")

Saved 1896 audio files to /home/vladislavstankov/Projects/Python/overlap-annotations/selected_audios


In [5]:
tsv_df = anotations_pool.drop(columns=["audio"])
tsv_df.to_csv("../annotation_pool.tsv", sep="\t", index=False)
print(f"Saved annotation_pool.tsv with {len(tsv_df)} rows, columns: {list(tsv_df.columns)}")

Saved annotation_pool.tsv with 1896 rows, columns: ['start_overlap', 'end_overlap', 'Duration', 'origin', 'overlap_dur', 'rms_global', 'p80', 'prediction', 'is_bad', 'audio_path']


In [6]:
tsv_df

,start_overlap,end_overlap,Duration,origin,overlap_dur,rms_global,p80,prediction,is_bad,audio_path
18,717.127334,721.134126,4.006791,2014021211581212,0.094907,0.042391,0.051567,"Nakonec, včera jsme byli svědky při rozsáhlé d...",False,/home/vladislavstankov/Projects/Python/overlap...
38,613.521222,617.803056,4.281834,2014050710281042,0.169496,0.036923,0.052248,"Já jsem, kolegyně a kolegové.",False,/home/vladislavstankov/Projects/Python/overlap...
43,342.074703,346.672326,4.597623,2014051417381752,0.094454,0.036892,0.046102,pak vyvěduje ještě přilážku ministra zahraničí...,False,/home/vladislavstankov/Projects/Python/overlap...
54,361.704584,367.728353,6.023769,2014061309080922,0.100396,0.024288,0.020187,"Ano, takže by to byl druhý bod.",False,/home/vladislavstankov/Projects/Python/overlap...
59,604.000000,607.803056,3.803056,2014061909280942,0.063384,0.034029,0.047387,Ježíte potom v podrobné rozpravě přečtu na obr...,False,/home/vladislavstankov/Projects/Python/overlap...
...,...,...,...,...,...,...,...,...,...,...
14987,326.000000,329.633277,3.633277,2023031010181032,0.214601,0.064477,0.071895,"Ano, ano. Nechme to tak. Děkuji.",False,/home/vladislavstankov/Projects/Python/overlap...
15013,46.000000,57.521222,11.521222,2023051816481702,0.208998,0.100649,0.131199,"My se tady nebavíme o dětech, které jsou jako ...",False,/home/vladislavstankov/Projects/Python/overlap...
15024,721.629881,730.010187,8.380306,2023060213481402,0.174307,0.239944,0.295192,"A to v oblasti. Už je konec. Já si omlouvám, ž...",False,/home/vladislavstankov/Projects/Python/overlap...
15031,284.237691,287.534805,3.297114,2023061412081222,0.126429,0.136076,0.183570,"kritizující úřad, musím vás představit.",False,/home/vladislavstankov/Projects/Python/overlap...
